# Week 36, Day 3: Iterator

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w036d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
numbers = [10, 20, 30]
it = iter(numbers)
print(next(it), list(it), list(it))
print(list(numbers), list(numbers))
pairs = zip("ab", [1, 2])
print(list(pairs), list(pairs))

In [ ]:
things = ["rice", "oil", "salt"]

it = iter(things)              # ask the collection for an iterator
while True:
    try:
        x = next(it)           # ask the iterator for the next item
    except StopIteration:      # the iterator says there's nothing left
        break
    print(x)

In [ ]:
r = range(3)
print(iter(r) is iter(r))            # an iterable: a new iterator each time
z = zip("ab", "cd")
print(iter(z) is z)                  # an iterator: iter() gives back itself
print(list(r), list(r), list(z), list(z))

In [ ]:
class Countdown:
    def __init__(self, start):
        self.current = start

    def __iter__(self):
        return self

    def __next__(self):
        if self.current < 0:
            raise StopIteration
        value = self.current
        self.current -= 1
        return value

c = Countdown(3)
print(list(c), list(c))
for n in Countdown(2):
    print(n, end=" ")
print()

In [ ]:
from itertools import islice

class Squares:
    def __init__(self):
        self.n = 0
    def __iter__(self):
        return self
    def __next__(self):
        self.n += 1
        return self.n * self.n

print(list(islice(Squares(), 6)))
for sq in Squares():
    if sq > 50:
        print("first square over 50:", sq)
        break

In [ ]:
class Playlist:
    def __init__(self, songs):
        self._songs = list(songs)

    def __iter__(self):
        return PlaylistIterator(self._songs)

class PlaylistIterator:
    def __init__(self, songs):
        self._songs = songs
        self._i = 0

    def __iter__(self):
        return self

    def __next__(self):
        if self._i >= len(self._songs):
            raise StopIteration
        song = self._songs[self._i]
        self._i += 1
        return song

p = Playlist(["Ye", "Essence", "Calm Down"])
print(list(p), list(p))
print([(a, b) for a in p for b in p if a < b][:2])     # two loops at once, each with its own iterator

In [ ]:
class TreeIterator:
    """Yields (depth, name) for a tree of nested dictionaries, depth-first, children in order."""

    def __init__(self, tree):
        self._stack = [(0, name, children) for name, children in reversed(list(tree.items()))]

    def __iter__(self):
        return self

    def __next__(self):
        if not self._stack:
            raise StopIteration
        depth, name, children = self._stack.pop()
        for child, grandchildren in reversed(list(children.items())):
            self._stack.append((depth + 1, child, grandchildren))
        return depth, name

shop = {"Food": {"Grains": {"Rice": {}, "Garri": {}}, "Drinks": {"Zobo": {}}}, "Home": {"Cleaning": {}}}
for depth, name in TreeIterator(shop):
    print("  " * depth + name)

> **Key idea.** Iterator (Gang of Four): access items one at a time without exposing how they're stored. A `for` loop calls `iter()` to get an iterator and `next()` until it raises `StopIteration`. An iterable's `__iter__` returns a new iterator each time, so it can be looped over repeatedly; an iterator has `__next__` and returns itself from `__iter__`, so it's used up after one pass (like `zip`, `map` and `enumerate`). Write an iterator class to compute items on demand, to walk a structure such as a tree with an explicit stack, or to produce endless sequences (stopped with `break` or `itertools.islice`). Phase 5's generators make these shorter.

### Exercise: An iterator of steps

Write an iterator class `Steps(start, stop, step)` that produces `start`, `start + step`, ... while the value is less than `stop` (for a positive step) or greater than `stop` (for a negative step), like `range` but for floats too. A step of 0 raises `ValueError` when it's made. Being an iterator, it's used up after one pass.

In [ ]:
class Steps:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_steps():\n    \"steps\"\n    S = ns[\"Steps\"]\n    assert list(S(0, 1, 0.25)) == [0, 0.25, 0.5, 0.75] and list(S(10, 0, -3)) == [10, 7, 4, 1] and list(S(5, 5, 1)) == []\n    s = S(1, 4, 1)\n    assert iter(s) is s and next(s) == 1 and list(s) == [2, 3] and list(s) == []\n    try:\n        S(0, 1, 0)\n        assert False, \"a step of 0 should raise ValueError\"\n    except ValueError:\n        pass")

### Exercise: Pages you can loop over again

Write an **iterable** `Pages(items, size)` that gives the items in lists of `size` (the last page may be shorter), with its own iterator class `PageIterator`. Looping over a `Pages` twice must give the pages twice. It also has `__len__`, the number of pages. A size below 1 raises `ValueError`.

In [ ]:
___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_pages():\n    \"pages\"\n    P, PI = ns[\"Pages\"], ns[\"PageIterator\"]\n    p = P(range(7), 3)\n    assert list(p) == [[0, 1, 2], [3, 4, 5], [6]] and list(p) == [[0, 1, 2], [3, 4, 5], [6]] and len(p) == 3\n    assert isinstance(iter(p), PI) and iter(p) is not iter(p) and list(P([], 5)) == [] and len(P([], 5)) == 0\n    it = iter(P(\"abcd\", 2))\n    assert next(it) == [\"a\", \"b\"] and next(it) == [\"c\", \"d\"] and next(it, None) is None\n    try:\n        P([1], 0)\n        assert False, \"size 0 should raise ValueError\"\n    except ValueError:\n        pass")

### Exercise: Taking turns

Write an iterator `RoundRobin(*iterables)` that takes one item from each iterable in turn, skipping any that have run out, until all have: `RoundRobin("ab", [1, 2, 3], "z")` gives `a 1 z b 2 3`. It must work for iterators too (such as `iter(...)` or `zip` objects), so use `iter()` and `next()` on each, not indexes.

In [ ]:
class RoundRobin:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_rr():\n    \"taking turns\"\n    R = ns[\"RoundRobin\"]\n    assert list(R(\"ab\", [1, 2, 3], \"z\")) == [\"a\", 1, \"z\", \"b\", 2, 3] and list(R()) == [] and list(R([], \"x\")) == [\"x\"]\n    assert list(R(iter([1, 2]), map(str.upper, \"ab\"))) == [1, \"A\", 2, \"B\"]")

### Exercise: Walking a folder tree

A folder tree is a dictionary: each key is a name, and its value is either another dictionary (a folder) or a number (a file's size in bytes). Write an iterator class `Files(tree)` that produces `(path, size)` pairs for every **file**, depth-first, in the order the keys appear, with paths joined by `/` (like `"docs/cv.txt"`). Use an explicit stack, not recursion.

In [ ]:
class Files:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_files():\n    \"every file\"\n    F = ns[\"Files\"]\n    tree = {\"docs\": {\"cv.txt\": 1200, \"notes\": {\"a.md\": 10, \"b.md\": 20}}, \"photo.jpg\": 5000, \"empty\": {}}\n    assert list(F(tree)) == [(\"docs/cv.txt\", 1200), (\"docs/notes/a.md\", 10), (\"docs/notes/b.md\", 20), (\"photo.jpg\", 5000)]\n    assert sum(size for _, size in F(tree)) == 6230 and list(F({})) == []\n    f = F({\"x\": 1})\n    assert iter(f) is f and next(f) == (\"x\", 1) and next(f, None) is None")

### Bug hunt: Only once

`Shelf` returns **itself** from `__iter__` and keeps its position, so the second loop finds nothing: this prints `['Aké', 'Kindred'] []`. Make `Shelf` an iterable that hands out a fresh iterator each time, so it prints `['Aké', 'Kindred'] ['Aké', 'Kindred']`.

In [ ]:
class Shelf:
    def __init__(self, titles):
        self.titles = list(titles)
        self.i = 0

    def __iter__(self):
        return self

    def __next__(self):
        if self.i >= len(self.titles):
            raise StopIteration
        self.i += 1
        return self.titles[self.i - 1]

s = Shelf(["Aké", "Kindred"])
print(list(s), list(s))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"again and again\"\n    expect_output(\"['Ak\u00e9', 'Kindred'] ['Ak\u00e9', 'Kindred']\")\n    s = ns[\"Shelf\"]([\"a\"])\n    assert iter(s) is not iter(s)")

### Bug hunt: The wrong way to finish

The iterator signals the end by raising `IndexError` instead of `StopIteration`, so the `for` loop crashes instead of finishing. Fix `__next__` so this prints `3 2 1` and then `lift off`.

In [ ]:
class Countdown:
    def __init__(self, start):
        self.n = start

    def __iter__(self):
        return self

    def __next__(self):
        if self.n == 0:
            raise IndexError("no more numbers")
        self.n -= 1
        return self.n + 1

for n in Countdown(3):
    print(n, end=" ")
print()
print("lift off")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"finishing properly\"\n    expect_output(\"3 2 1\\nlift off\")")

### Bug hunt: A zip used twice

The pairs are made once with `zip` and used twice, so the second use finds nothing and the report says the total is 0: this prints `Ada 70`, `Bo 85` and then `total 0`. Fix the line that makes `pairs` so it can be used twice, and the program prints `total 155`.

In [ ]:
names = ["Ada", "Bo"]
marks = [70, 85]
pairs = zip(names, marks)
for name, mark in pairs:
    print(name, mark)
print("total", sum(mark for _, mark in pairs))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"used twice\"\n    expect_output(\"Ada 70\\nBo 85\\ntotal 155\")")

### Puzzle: An iterator class

Arrange the lines into an iterator that produces the letters of a word in reverse. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
it = iter("abc")
first = next(it)
rest = list(it)
again = list(it)
m = map(str.upper, "xy")
d = next(iter([]), "empty")

p = ___   # first
q = ___   # rest
r = ___   # again
s = ___   # (list(m), list(m))
t = ___   # d

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert (p, q, r) == (\"a\", [\"b\", \"c\"], []), \"p to r\"\n    assert s == ([\"X\", \"Y\"], []) and t == \"empty\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: working days

Write two classes for stepping through calendar dates:

- `DateRange(start, end, step=1)`: an **iterable** of `datetime.date`s from `start` up to and including `end`, every `step` days (a step below 1 raises `ValueError`). It can be looped over repeatedly (its `__iter__` returns a new `DateRangeIterator`), has `__len__` (how many dates), and supports `in` for a date (worked out without looping: inside the range and a whole number of steps from `start`).
- `WorkingDays(start, holidays=())`: an endless **iterator** of the dates from `start` onwards (including `start` itself if it counts) that aren't Saturdays, Sundays or in `holidays`.

Then write `add_working_days(start, n, holidays=())`, the date `n` working days after `start` (counting from the next working day after `start`; `n` of 0 gives `start`), using `WorkingDays`.

In [ ]:
from datetime import date, timedelta

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_range():\n    \"date ranges\"\n    from datetime import date as d\n    D = ns[\"DateRange\"]\n    r = D(d(2026, 11, 1), d(2026, 11, 10), 3)\n    assert list(r) == [d(2026, 11, 1), d(2026, 11, 4), d(2026, 11, 7), d(2026, 11, 10)] and list(r) == list(r) and len(r) == 4\n    assert d(2026, 11, 7) in r and d(2026, 11, 8) not in r and d(2026, 10, 29) not in r and d(2026, 11, 13) not in r\n    assert len(D(d(2026, 11, 1), d(2026, 11, 1))) == 1 and list(D(d(2026, 11, 2), d(2026, 11, 1))) == [] and len(D(d(2026, 11, 2), d(2026, 11, 1))) == 0\n    assert type(iter(r)).__name__ == \"DateRangeIterator\"\n    try:\n        D(d(2026, 1, 1), d(2026, 2, 1), 0)\n        assert False, \"step 0 should raise ValueError\"\n    except ValueError:\n        pass\ndef test_working():\n    \"working days\"\n    from datetime import date as d\n    from itertools import islice\n    W = ns[\"WorkingDays\"]\n    w = W(d(2026, 12, 24), holidays={d(2026, 12, 25), d(2026, 12, 28)})\n    assert list(islice(w, 4)) == [d(2026, 12, 24), d(2026, 12, 29), d(2026, 12, 30), d(2026, 12, 31)] and iter(w) is w\n    assert next(W(d(2026, 11, 7))) == d(2026, 11, 9)\n    assert add_working_days(d(2026, 11, 6), 1) == d(2026, 11, 9) and add_working_days(d(2026, 11, 6), 0) == d(2026, 11, 6)\n    assert add_working_days(d(2026, 12, 23), 3, {d(2026, 12, 25), d(2026, 12, 28)}) == d(2026, 12, 30)")

## Warm-up

A list can be looped over again and again, but `it` (made by `iter`) and the `zip` object are used up after one pass: the second `list(...)` finds nothing left. Week 9 warned you about that with `zip` and `enumerate`. Today explains why, and how to write both kinds of object yourself.

## What a for loop really does

The Gang of Four describe the Iterator pattern as: "provide a way to access the elements of an aggregate object sequentially without exposing its underlying representation." Python builds it into the language. This loop:

```python
for x in things:
    print(x)
```

does exactly this:

- `iter(things)` calls `things.__iter__()`, which returns an **iterator**: an object that remembers where it's got to.
- `next(it)` calls `it.__next__()`, which returns the next item, or raises **`StopIteration`** when there are none left. `StopIteration` isn't an error: it's how an iterator says "finished", and `for` catches it for you.
- `next(it, default)` returns `default` instead of raising, which you used in Week 18 to find the first match.

Everything that works with `for` works this way: `list(...)`, `sum`, `sorted`, `in`, unpacking, comprehensions.

## Iterables and iterators

Two roles, often confused:

| | Iterable | Iterator |
|---|---|---|
| What it is | something you can loop over | the thing doing the looping, remembering its place |
| Must have | `__iter__`, returning a **new** iterator | `__next__`, and `__iter__` returning **itself** |
| Loop over it twice? | yes: each loop gets a fresh iterator | no: once it's used up, it stays used up |
| Examples | lists, strings, dictionaries, ranges, files' contents as stored data | `iter(list)`, `zip`, `enumerate`, `map`, open files (as you read them) |

An iterator's `__iter__` returns itself so that an iterator can be used anywhere an iterable can (`for x in zip(...)`), but that's also why it's one-shot: both loops get the same, already-advanced object.

Iterators have two big advantages. They're **lazy**: they work out each item only when asked, so they can represent huge or even endless sequences without storing them. And they **hide the structure**: code looping over a tree, a file or a database result doesn't need to know how the items are stored.

## Writing an iterator

An iterator class needs `__next__`, which returns the next item or raises `StopIteration`, and `__iter__`, which returns `self`:

Iterators can also be **infinite**: one that never raises `StopIteration` must be stopped by the loop using it (with `break`, or by taking only some items). `itertools.islice(iterator, n)` takes the first `n` items of any iterator:

## Fresh iterators every time

Usually you want a collection that can be looped over **repeatedly**, so the collection should be an **iterable** whose `__iter__` makes a **new iterator object** each time. The iterator keeps the position; the collection keeps the data:

Week 31's shortcut, `return iter(self._songs)`, does the same thing using the list's own iterator. Write your own iterator class when the order or the items need working out (skipping, transforming, walking a structure), not just handing out a stored list.

## Walking a structure

The real strength of iterators shows with structures that aren't simple lists. Here's an iterator that walks a tree of categories **depth-first** (each item, then everything under it, before its next sibling), using an explicit stack (Week 8) instead of recursion, so it can stop and resume between items:

Children are pushed in **reverse** so that the first child is popped first. Swapping the stack for a queue (taking from the front with `collections.deque`'s `popleft`) would give a **breadth-first** walk instead: every item at depth 0, then depth 1, and so on. Phase 4 studies these two orders properly.

## Practice

## Quiz

## Daily challenge

## Before you go

Look at a class of your own that holds a collection and decide whether it should be an iterable (looped over many times) or an iterator (used once), then check what its `__iter__` returns. Tomorrow: stepping back from patterns to whole programs, with layered architecture.